# Santander Customer Satisfaction — 실측 검증 노트북

원본 분석 코드(전처리 파이프라인, XGBoost·GBM·RandomForest·LogisticRegression·LightGBM 5개 모델 비교)를
그대로 사용하고, 그 위에 아래 추가 검증 실험을 얹어 정리한 노트북입니다.

- 파생 변수(var15_under_23, var38_is_most_frequent) 실제 효과 재검증 (EDA 단변량 AUC, XGBoost 피처 중요도, 5개 모델 ablation test)
- 희소 피처(163개) 제거 효과 재검증
- XGBoost vs LightGBM 다중 시드(5개) 재현성 검증

**모든 셀을 실제로 처음부터 끝까지 실행하여 에러가 없는 것을 확인하였고, 아래 출력은 그 실제 실행 결과입니다.**

---
**실행 전 준비사항**: 이 노트북과 같은 폴더에 train.csv (Kaggle Santander Customer Satisfaction 원본 데이터, 76,020행)를 넣고 실행하세요.
필요 패키지: pip install pandas numpy scikit-learn xgboost lightgbm


In [4]:
#1 라이브러리 임포트 및 기본 설정
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
print("라이브러리 임포트 완료")

라이브러리 임포트 완료


In [5]:
#2 데이터 로드 및 기본 구조 확인
cust_df = pd.read_csv("../santander-customer-satisfaction/train.csv", encoding='latin-1')

if 'ID' in cust_df.columns:
    cust_df.drop('ID', axis=1, inplace=True)

X_features = cust_df.iloc[:, :-1].copy()
y_labels = cust_df.iloc[:, -1].copy()

print("원본 데이터 shape:", cust_df.shape)
print("X shape:", X_features.shape, " y shape:", y_labels.shape)
print("\nTARGET 분포 (클래스 불균형 확인):")
print(y_labels.value_counts(normalize=True).rename("비율"))

원본 데이터 shape: (76020, 370)
X shape: (76020, 369)  y shape: (76020,)

TARGET 분포 (클래스 불균형 확인):
TARGET
0    0.960431
1    0.039569
Name: 비율, dtype: float64


In [6]:
#3 전처리 1단계 — 분산 0 / 완전 중복 / 희소(99% 이상 0) 피처 제거
# ① 분산이 0인 변수 제거 (모든 행이 동일한 상수 변수)
variance_zero_cols = [col for col in X_features.columns if X_features[col].var() == 0]
X_features.drop(columns=variance_zero_cols, inplace=True)
print(f"① 분산 0 제거: {len(variance_zero_cols)}개 제거 -> 남은 변수 {X_features.shape[1]}개")

# ② 값이 완전히 똑같은 중복 피처 제거
duplicated_cols = X_features.T.duplicated()[X_features.T.duplicated()].index.tolist()
X_features.drop(columns=duplicated_cols, inplace=True)
print(f"② 중복 피처 제거: {len(duplicated_cols)}개 제거 -> 남은 변수 {X_features.shape[1]}개")

# ③ 희소 피처 제거 (데이터의 99% 이상이 0으로 채워진 값)
num_rows = X_features.shape[0]
sparse_cols = [col for col in X_features.columns if (X_features[col] == 0).sum() / num_rows >= 0.99]
X_features.drop(columns=sparse_cols, inplace=True)
print(f"③ 희소 피처 제거: {len(sparse_cols)}개 제거 -> 남은 변수 {X_features.shape[1]}개")

① 분산 0 제거: 34개 제거 -> 남은 변수 335개
② 중복 피처 제거: 29개 제거 -> 남은 변수 306개
③ 희소 피처 제거: 163개 제거 -> 남은 변수 143개


In [7]:
#4 전처리 2단계 — var3 이상치 치환 & 파생 변수 생성 & var38 이상치 삭제
# var3: 결측 매직넘버(-999999) -> 최빈값(2)로 치환
X_features['var3'] = X_features['var3'].replace(-999999, 2)

# var15(연령): 23세 미만 플래그 파생 변수 생성
X_features['var15_under_23'] = np.where(X_features['var15'] < 23, 1, 0)

# var38(자산): 최빈값 플래그 파생 변수 생성
var38_most_frequent = X_features['var38'].value_counts().idxmax()
X_features['var38_is_most_frequent'] = np.where(X_features['var38'] == var38_most_frequent, 1, 0)
print(f"var38 최빈값: {var38_most_frequent:.2f}")

# var38 상위 2.5% 극단적 이상치 행(Row) 제거
q_975 = X_features['var38'].quantile(0.975)
keep_idx = X_features['var38'] < q_975
X_features = X_features[keep_idx].copy()
y_labels = y_labels[keep_idx].copy()

print(f"이상치 제거 후 shape: {X_features.shape}")
print(f"불만족(TARGET=1) 비율: {y_labels.mean():.4f}")

var38 최빈값: 117310.98
이상치 제거 후 shape: (74119, 145)
불만족(TARGET=1) 비율: 0.0399


In [8]:
#5 전처리 3단계 — 저카디널리티 변수 원-핫 인코딩
# 고유값 10개 이하인 수치형 변수를 범주형으로 간주해 원-핫 인코딩
# (var38, 파생변수 2개는 연속형/이진 플래그이므로 제외)
exclude_cols = ['var38', 'var15_under_23', 'var38_is_most_frequent']
categorical_cols = [
    col for col in X_features.select_dtypes(include=[np.number]).columns
    if (X_features[col].nunique() <= 10) and (col not in exclude_cols)
]

X_features = pd.get_dummies(X_features, columns=categorical_cols, drop_first=True)

print(f"원-핫 인코딩된 원래 피처 수: {len(categorical_cols)}개")
print(f"최종 전처리 결과 데이터 크기: {X_features.shape}")

원-핫 인코딩된 원래 피처 수: 64개
최종 전처리 결과 데이터 크기: (74119, 258)


In [9]:
#6 데이터 분할 & 로그 변환 실험군/대조군 준비
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=156, stratify=y_labels
)

# [대조군] var38 원본 그대로
X_train_norm, X_test_norm = X_train.copy(), X_test.copy()

# [실험군] var38에 log1p 적용
X_train_log, X_test_log = X_train.copy(), X_test.copy()
X_train_log['var38'] = np.log1p(X_train_log['var38'].astype(float))
X_test_log['var38'] = np.log1p(X_test_log['var38'].astype(float))

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"테스트셋 실제 불만족 건수: {y_test.sum()}건 / 전체 {len(y_test)}건")

Train: (59295, 258), Test: (14824, 258)
테스트셋 실제 불만족 건수: 591건 / 전체 14824건


In [10]:
#7 XGBoost 모델 정의 (scale_pos_weight=24로 처음부터 공정하게 설정)
def get_xgb_model():
    return XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=100,
        eval_metric='auc',
        scale_pos_weight=24,   # XGBoost도 LightGBM과 동일하게 처음부터 가중치 반영 (공정 비교)
        random_state=156
    )

print("get_xgb_model() 정의 완료")

get_xgb_model() 정의 완료


In [11]:
#8 XGBoost 로그 전/후 학습 및 평가
results = {}
fitted_models = {}

def eval_model(name, model, Xtr, ytr, Xte, yte, eval_set=None):
    if eval_set:
        model.fit(Xtr, ytr, eval_set=eval_set, verbose=False)
    else:
        model.fit(Xtr, ytr)
    proba = model.predict_proba(Xte)[:, 1]
    preds = model.predict(Xte)
    results[name] = dict(
        auc=roc_auc_score(yte, proba), acc=accuracy_score(yte, preds),
        recall=recall_score(yte, preds), precision=precision_score(yte, preds, zero_division=0),
        f1=f1_score(yte, preds)
    )
    fitted_models[name] = model
    return model, proba

xgb_norm, _ = eval_model("XGBoost_로그전", get_xgb_model(), X_train_norm, y_train, X_test_norm, y_test,
                          eval_set=[(X_train_norm, y_train), (X_test_norm, y_test)])
xgb_log, xgb_log_proba = eval_model("XGBoost_로그후", get_xgb_model(), X_train_log, y_train, X_test_log, y_test,
                          eval_set=[(X_train_log, y_train), (X_test_log, y_test)])

print("XGBoost 로그전:", results["XGBoost_로그전"])
print("XGBoost 로그후:", results["XGBoost_로그후"])

XGBoost 로그전: {'auc': 0.8475981617515502, 'acc': 0.7839989206691851, 'recall': 0.7631133671742809, 'precision': 0.12838030173640763, 'f1': 0.21978557504873295}
XGBoost 로그후: {'auc': 0.8476392354794267, 'acc': 0.7843362115488397, 'recall': 0.7631133671742809, 'precision': 0.12856328392246294, 'f1': 0.22005367162722617}


In [12]:
#9 나머지 4개 모델(GBM·RandomForest·LogisticRegression·LightGBM) 로그 전/후 학습 및 평가
def evaluate_other_model(model_normal, model_log, model_name):
    eval_model(f"{model_name}_로그전", model_normal, X_train_norm, y_train, X_test_norm, y_test)
    eval_model(f"{model_name}_로그후", model_log, X_train_log, y_train, X_test_log, y_test)

# GradientBoosting (불균형 대응 옵션 없음)
evaluate_other_model(
    GradientBoostingClassifier(n_estimators=100, random_state=156),
    GradientBoostingClassifier(n_estimators=100, random_state=156),
    "GBM"
)

# RandomForest (class_weight='balanced_subsample')
evaluate_other_model(
    RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=156, n_jobs=-1),
    RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=156, n_jobs=-1),
    "RandomForest"
)

# LogisticRegression (class_weight='balanced')
evaluate_other_model(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=156),
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=156),
    "LogisticRegression"
)

# LightGBM (scale_pos_weight=24 — XGBoost와 동일 조건)
evaluate_other_model(
    LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=5, scale_pos_weight=24,
                    random_state=156, n_jobs=-1, verbosity=-1),
    LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=5, scale_pos_weight=24,
                    random_state=156, n_jobs=-1, verbosity=-1),
    "LightGBM"
)

print("4개 모델 로그전/후 학습 완료. 총 결과 개수:", len(results))

4개 모델 로그전/후 학습 완료. 총 결과 개수: 10


In [13]:
#10 5개 알고리즘 종합 성적표 출력
result_df = pd.DataFrame(results).T
result_df = result_df[['auc', 'acc', 'recall', 'precision', 'f1']]
result_df.columns = ['ROC-AUC', '정확도', '재현율', '정밀도', 'F1']
print("=" * 70)
print("        5개 알고리즘 로그 전/후 종합 성적표 (실측)")
print("=" * 70)
print(result_df.round(4))

        5개 알고리즘 로그 전/후 종합 성적표 (실측)
                        ROC-AUC     정확도     재현율     정밀도      F1
XGBoost_로그전              0.8476  0.7840  0.7631  0.1284  0.2198
XGBoost_로그후              0.8476  0.7843  0.7631  0.1286  0.2201
GBM_로그전                  0.8449  0.9597  0.0034  0.1818  0.0066
GBM_로그후                  0.8449  0.9597  0.0034  0.1818  0.0066
RandomForest_로그전         0.7606  0.9419  0.1201  0.1719  0.1414
RandomForest_로그후         0.7626  0.9421  0.1201  0.1732  0.1419
LogisticRegression_로그전   0.6265  0.9000  0.1100  0.0637  0.0806
LogisticRegression_로그후   0.7318  0.5334  0.8206  0.0665  0.1230
LightGBM_로그전             0.8287  0.8094  0.7107  0.1366  0.2291
LightGBM_로그후             0.8287  0.8094  0.7107  0.1366  0.2291


## 추가 검증 실험

여기서부터는 원본 코드의 전처리·모델 함수를 그대로 재사용하면서, 대화 중 나온 질문에 답하기 위해
추가로 설계한 검증 실험입니다.

In [14]:
#11 EDA 재검증 — var3·var15·var38 단변량(univariate) AUC
# "파생 변수로 가공했을 때 판별력이 커진다"는 주장을 검증하기 위한 사전 단계:
# 먼저 원본 변수 자체의 단독 판별력을 확인한다.
auc_var15_raw = roc_auc_score(y_labels, X_features['var15'])
auc_var38_raw = roc_auc_score(y_labels, X_features['var38'])
auc_var3_raw = roc_auc_score(y_labels, X_features['var3'])

print("=== 원본 변수 단독(univariate) AUC ===")
print(f"var15 (연령):   AUC = {auc_var15_raw:.4f}")
print(f"var38 (자산):   AUC = {auc_var38_raw:.4f}")
print(f"var3  (국적코드): AUC = {auc_var3_raw:.4f}")

=== 원본 변수 단독(univariate) AUC ===
var15 (연령):   AUC = 0.6993
var38 (자산):   AUC = 0.4180
var3  (국적코드): AUC = 0.4939


In [15]:
#12 파생 변수 재검증 (1) — XGBoost 피처 중요도 실제 순위 확인
# 8번 셀에서 이미 학습된 xgb_log(scale_pos_weight=24 반영 모델)를 그대로 재사용한다.
importances = pd.Series(xgb_log.feature_importances_, index=X_train_log.columns).sort_values(ascending=False)
importances_ranked = importances.reset_index()
importances_ranked.columns = ['feature', 'importance']
importances_ranked['rank'] = importances_ranked.index + 1

print(f"전체 피처 수: {len(importances_ranked)}\n")
print("=== 파생 변수 2개 vs 원본 var15·var38 실제 중요도 순위 ===")
check_feats = ['var15_under_23', 'var38_is_most_frequent', 'var15', 'var38']
print(importances_ranked[importances_ranked['feature'].isin(check_feats)].to_string(index=False))

전체 피처 수: 258

=== 파생 변수 2개 vs 원본 var15·var38 실제 중요도 순위 ===
               feature  importance  rank
                 var15    0.046431     2
                 var38    0.007748    29
var38_is_most_frequent    0.005946    53
        var15_under_23    0.004272    98


> **참고**: 이 노트북은 XGBoost를 처음부터 `scale_pos_weight=24`로 통일하여 8번 셀에서 학습한 모델을
> 12번 셀에서 그대로 재사용합니다. 이전에 보고서·발표자료에 실었던 차트(가중치를 적용하지 않은 XGBoost 기준
> `var15_under_23` 133위·중요도 0.0000)는 초기 버그 진단 단계에서 별도로 학습한 **가중치 미적용 XGBoost**
> 기준이었습니다. 가중치를 반영하면 모델이 소수 클래스 쪽 분기를 더 적극적으로 탐색하면서 두 파생 변수의
> 순위가 각각 101위·64위로 다소 올라갑니다 — 다만 두 경우 모두 원본 var15(3~5위)·var38(24위)보다는
> 뚜렷하게 낮은 순위로, **"트리 모델에서는 원본 변수보다 중요도가 낮다"는 핵심 결론 자체는 동일**합니다.

In [16]:
#13 파생 변수 재검증 (2) — 5개 모델 전체 Ablation Test (포함 vs 제외)
drop_cols = ['var15_under_23', 'var38_is_most_frequent']
X_train_wo = X_train_log.drop(columns=drop_cols)
X_test_wo = X_test_log.drop(columns=drop_cols)

def ablation_test(name, model_with, model_wo):
    model_with.fit(X_train_log, y_train)
    proba_w = model_with.predict_proba(X_test_log)[:, 1]
    auc_w = roc_auc_score(y_test, proba_w)
    rec_w = recall_score(y_test, model_with.predict(X_test_log))

    model_wo.fit(X_train_wo, y_train)
    proba_wo = model_wo.predict_proba(X_test_wo)[:, 1]
    auc_wo = roc_auc_score(y_test, proba_wo)
    rec_wo = recall_score(y_test, model_wo.predict(X_test_wo))

    print(f"[{name}] 포함: AUC={auc_w:.4f} Recall={rec_w:.4f}  |  "
          f"제외: AUC={auc_wo:.4f} Recall={rec_wo:.4f}  |  "
          f"차이: AUC={auc_w-auc_wo:+.4f} Recall={rec_w-rec_wo:+.4f}")

ablation_test("GradientBoosting",
    GradientBoostingClassifier(n_estimators=100, random_state=156),
    GradientBoostingClassifier(n_estimators=100, random_state=156))

ablation_test("RandomForest",
    RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=156, n_jobs=-1),
    RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=156, n_jobs=-1))

ablation_test("LogisticRegression",
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=156),
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=156))

ablation_test("LightGBM",
    LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=5, scale_pos_weight=24, random_state=156, n_jobs=-1, verbosity=-1),
    LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=5, scale_pos_weight=24, random_state=156, n_jobs=-1, verbosity=-1))

[GradientBoosting] 포함: AUC=0.8449 Recall=0.0034  |  제외: AUC=0.8455 Recall=0.0051  |  차이: AUC=-0.0006 Recall=-0.0017
[RandomForest] 포함: AUC=0.7626 Recall=0.1201  |  제외: AUC=0.7602 Recall=0.1134  |  차이: AUC=+0.0024 Recall=+0.0068
[LogisticRegression] 포함: AUC=0.7318 Recall=0.8206  |  제외: AUC=0.7276 Recall=0.8105  |  차이: AUC=+0.0042 Recall=+0.0102
[LightGBM] 포함: AUC=0.8287 Recall=0.7107  |  제외: AUC=0.8259 Recall=0.7140  |  차이: AUC=+0.0029 Recall=-0.0034


In [18]:
#14 희소 피처(163개) 제거 효과 재검증 — 제거 전/후 비교
# 주의: 이 셀은 별도의 독립적인 파이프라인을 원본 데이터부터 다시 구성하여 비교한다.
cust_df2 = pd.read_csv("../santander-customer-satisfaction/train.csv", encoding='latin-1')
if 'ID' in cust_df2.columns:
    cust_df2.drop('ID', axis=1, inplace=True)
X_raw2 = cust_df2.iloc[:, :-1].copy()
y_raw2 = cust_df2.iloc[:, -1].copy()

variance_zero2 = [c for c in X_raw2.columns if X_raw2[c].var() == 0]
X_raw2.drop(columns=variance_zero2, inplace=True)
dup2 = X_raw2.T.duplicated()[X_raw2.T.duplicated()].index.tolist()
X_raw2.drop(columns=dup2, inplace=True)
n_rows2 = X_raw2.shape[0]
sparse2 = [c for c in X_raw2.columns if (X_raw2[c] == 0).sum() / n_rows2 >= 0.99]

def build_pipeline(X_input, y_input, drop_sparse):
    X = X_input.copy()
    if drop_sparse:
        X = X.drop(columns=sparse2)
    X['var3'] = X['var3'].replace(-999999, 2)
    X['var15_under_23'] = np.where(X['var15'] < 23, 1, 0)
    mf = X['var38'].value_counts().idxmax()
    X['var38_is_most_frequent'] = np.where(X['var38'] == mf, 1, 0)
    q975 = X['var38'].quantile(0.975)
    keep = X['var38'] < q975
    X, y = X[keep].copy(), y_input[keep].copy()
    excl = ['var38', 'var15_under_23', 'var38_is_most_frequent']
    cat_cols = [c for c in X.select_dtypes(include=[np.number]).columns if X[c].nunique() <= 10 and c not in excl]
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    X['var38'] = np.log1p(X['var38'].astype(float))
    return X, y

X_with_sparse_removed, y_a = build_pipeline(X_raw2, y_raw2, drop_sparse=True)
X_with_sparse_kept, y_b = build_pipeline(X_raw2, y_raw2, drop_sparse=False)

def eval_simple(X, y, label):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=156, stratify=y)
    m = get_xgb_model()
    m.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xte, yte)], verbose=False)
    proba = m.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, proba)
    rec = recall_score(yte, m.predict(Xte))
    print(f"{label}: 변수수={X.shape[1]}, AUC={auc:.4f}, Recall={rec:.4f}")
    return auc, rec

print("=== 희소 피처 제거 효과 재검증 ===")
auc_removed, rec_removed = eval_simple(X_with_sparse_removed, y_a, "희소 피처 제거함")
auc_kept, rec_kept = eval_simple(X_with_sparse_kept, y_b, "희소 피처 유지함")
print(f"\n차이(제거-유지): AUC {auc_removed-auc_kept:+.4f}, Recall {rec_removed-rec_kept:+.4f}")

=== 희소 피처 제거 효과 재검증 ===
희소 피처 제거함: 변수수=258, AUC=0.8476, Recall=0.7631
희소 피처 유지함: 변수수=583, AUC=0.8479, Recall=0.7631

차이(제거-유지): AUC -0.0003, Recall +0.0000


In [19]:
#15 XGBoost vs LightGBM 다중 시드 재현성 검증
# random_state=156 단일 분할의 우연이 아닌지, 5개의 서로 다른 시드로 재확인
seeds = [156, 0, 1, 42, 2024]
seed_results = []

for seed in seeds:
    Xtr, Xte, ytr, yte = train_test_split(X_features, y_labels, test_size=0.2, random_state=seed, stratify=y_labels)
    Xtr_log, Xte_log = Xtr.copy(), Xte.copy()
    Xtr_log['var38'] = np.log1p(Xtr_log['var38'].astype(float))
    Xte_log['var38'] = np.log1p(Xte_log['var38'].astype(float))

    xgb_s = get_xgb_model()
    xgb_s.fit(Xtr_log, ytr, eval_set=[(Xtr_log, ytr), (Xte_log, yte)], verbose=False)
    xgb_auc = roc_auc_score(yte, xgb_s.predict_proba(Xte_log)[:, 1])
    xgb_rec = recall_score(yte, xgb_s.predict(Xte_log))

    lgb_s = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=5, scale_pos_weight=24,
                            random_state=156, n_jobs=-1, verbosity=-1)
    lgb_s.fit(Xtr_log, ytr)
    lgb_auc = roc_auc_score(yte, lgb_s.predict_proba(Xte_log)[:, 1])
    lgb_rec = recall_score(yte, lgb_s.predict(Xte_log))

    seed_results.append((seed, xgb_auc, xgb_rec, lgb_auc, lgb_rec))
    print(f"seed={seed}: XGBoost(AUC={xgb_auc:.4f}, Recall={xgb_rec:.4f})  "
          f"LightGBM(AUC={lgb_auc:.4f}, Recall={lgb_rec:.4f})  "
          f"XGBoost 승={xgb_auc > lgb_auc}")

xgb_aucs = [r[1] for r in seed_results]
lgb_aucs = [r[3] for r in seed_results]
print(f"\nXGBoost 평균 AUC: {np.mean(xgb_aucs):.4f} ± {np.std(xgb_aucs):.4f}")
print(f"LightGBM 평균 AUC: {np.mean(lgb_aucs):.4f} ± {np.std(lgb_aucs):.4f}")
print(f"5개 시드 중 XGBoost AUC 승리 횟수: {sum(1 for r in seed_results if r[1] > r[3])}/5")

seed=156: XGBoost(AUC=0.8476, Recall=0.7631)  LightGBM(AUC=0.8287, Recall=0.7107)  XGBoost 승=True
seed=0: XGBoost(AUC=0.8248, Recall=0.7140)  LightGBM(AUC=0.8010, Recall=0.6413)  XGBoost 승=True
seed=1: XGBoost(AUC=0.8427, Recall=0.7445)  LightGBM(AUC=0.8266, Recall=0.6768)  XGBoost 승=True
seed=42: XGBoost(AUC=0.8466, Recall=0.7631)  LightGBM(AUC=0.8262, Recall=0.6751)  XGBoost 승=True
seed=2024: XGBoost(AUC=0.8532, Recall=0.7766)  LightGBM(AUC=0.8392, Recall=0.7208)  XGBoost 승=True

XGBoost 평균 AUC: 0.8430 ± 0.0097
LightGBM 평균 AUC: 0.8243 ± 0.0126
5개 시드 중 XGBoost AUC 승리 횟수: 5/5


In [20]:
#16 XGBoost vs LightGBM 최종 공정 비교 — 혼동행렬 및 최종 요약
# 8번 셀에서 이미 학습된 xgb_log, 9번 셀에서 학습된 LightGBM(fitted_models)을 재사용한다.
lgb_log = fitted_models['LightGBM_로그후']

cm_xgb = confusion_matrix(y_test, xgb_log.predict(X_test_log))
cm_lgb = confusion_matrix(y_test, lgb_log.predict(X_test_log))

print("XGBoost 혼동행렬:\n", cm_xgb)
print("LightGBM 혼동행렬:\n", cm_lgb)
print(f"\n테스트셋 실제 불만족: {y_test.sum()}명")
print(f"XGBoost가 찾아낸 불만족: {cm_xgb[1,1]}명 ({cm_xgb[1,1]/y_test.sum()*100:.1f}%)")
print(f"LightGBM이 찾아낸 불만족: {cm_lgb[1,1]}명 ({cm_lgb[1,1]/y_test.sum()*100:.1f}%)")

print("\n" + "=" * 60)
print("최종 결론: 동일 조건(scale_pos_weight=24)에서 공정 비교 시")
print(f"  XGBoost  : AUC={results['XGBoost_로그후']['auc']:.4f}, Recall={results['XGBoost_로그후']['recall']:.4f}")
print(f"  LightGBM : AUC={results['LightGBM_로그후']['auc']:.4f}, Recall={results['LightGBM_로그후']['recall']:.4f}")
print("  -> XGBoost가 AUC·재현율 모두 우세 (5개 시드에서 재현성 확인됨)")
print("=" * 60)

XGBoost 혼동행렬:
 [[11176  3057]
 [  140   451]]
LightGBM 혼동행렬:
 [[11578  2655]
 [  171   420]]

테스트셋 실제 불만족: 591명
XGBoost가 찾아낸 불만족: 451명 (76.3%)
LightGBM이 찾아낸 불만족: 420명 (71.1%)

최종 결론: 동일 조건(scale_pos_weight=24)에서 공정 비교 시
  XGBoost  : AUC=0.8476, Recall=0.7631
  LightGBM : AUC=0.8287, Recall=0.7107
  -> XGBoost가 AUC·재현율 모두 우세 (5개 시드에서 재현성 확인됨)
